<table>
<tr><td><img style="height: 150px;" src="images/geo_hydro1.jpg"></td>
<td bgcolor="#FFFFFF">
    <p style="font-size: xx-large; font-weight: 900; line-height: 100%">AG Dynamics of the Earth</p>
    <p style="font-size: large; color: rgba(0,0,0,0.5);">Juypter notebooks</p>
    <p style="font-size: large; color: rgba(0,0,0,0.5);">Georg Kaufmann</p>
    </td>
</tr>
</table>

# Machine learning: Polynomial regression 

----
*Georg Kaufmann,
Geophysics Section,
Institute of Geological Sciences,
Freie Universität Berlin,
Germany*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

We continue with **polynomial regression**, fitting a polynomial through data points.

## Create sample data set

We first create a sample data set, using a polynomial function $f(x)$, and shaking both the
$x$ and the $y$ coordinates with uniform random noise:

In [ ]:
def f(x):
    y = x**3
    return y

In [ ]:
n = 41
np.random.seed(1234)
x = np.sort(np.random.uniform(-2,2,n))
#x = np.linspace(-2,2,n)
np.random.seed(1234)
y = f(x) + np.random.uniform(-2,2,n)

print(x.shape,y.shape)

We plot the function. The cubic polynom can still be seen, but the random noise distorts it quite a bit.

In [ ]:
# plot data points
fig=plt.figure(figsize=(12.0, 6.0))
plt.plot(x,y,marker='o',linestyle='none',color='red',label='data')
plt.legend()
plt.show()

Next, we want to use the data set in a **machine learning** algorithm. As we have seen  before,
the **feature matrix** $X$ has to be a two-dimensional array, even in the case with just
**one** feature, the $x$ coordinate.

We therefore need to extend the data array into the 2nd dimension for machine-learning algorithms:

In [ ]:
#X = x[:, np.newaxis]
X=x.reshape(n,1)
print(X.shape)

## Excursion: Polynomial interpolation with `numpy`

With an excursion to the classical `numpy` package, we use the `polyfit` module from the `numpy` package
for fitting a polynimial of degree $degree$ through a set of data points $(x_i,y_i),i=1,n$:
$$
P_m(x) = a_0 + a_1 x + \dots a_m x^m
$$
The module `polyfit` returns the coefficients $a_m$.

In [ ]:
degree = 3
coefs = np.polyfit(x, y,degree)
x_test = np.linspace(-2,2,100)

fig=plt.figure(figsize=(12.0, 6.0))
plt.plot(x_test, f(x_test),linewidth=5,color='gray',label="true")
plt.plot(x_test, np.polyval(coefs, x_test), linewidth=2,color="blue",label='P$_{'+str(degree)+'}$(x)')
plt.plot(x,y,marker='o',linestyle='none',color='red',label='data')
plt.legend()
plt.show()

There is a nice way to get the polynomial, the *convenience function* `poly1d`:

In [ ]:
print (np.poly1d(coefs))

## Polynomial regression

In the machine learning universe, **polynomial regression** is achieved with a **linear regression**
tool, but with **non-linear** functions, our polynomials coefficients.

Here, we need to understand better, what is done.

Consider our independent data variable $x_i,i=1,n$, our **features**, and the dependent data 
variable $y_i(x_i)$, the **labels**.

1. ***Linear fit with line***

In the case of fitting a **linear function** through the data, the fitting function will be
$$
y_i = a_0 + a_1 x_i
$$
with the two coefficients $a_0$ and $a_1$ we want to know.

For our $n$ observations, we can formulate the problem as:
$$
\left[
\begin{array}{c}
y_1 \\ y_2 \\ \vdots \\ y_n
\end{array}
\right]
=
\left[
\begin{array}{cc}
1 & x_1 \\
1 & x_2 \\
& \vdots \\
1 & x_n
\end{array}
\right]
\left[
\begin{array}{c}
a_0 \\ a_1
\end{array}
\right]
$$
For each of the $n$ equations, we need to **calculate** the coefficients of the matrix,
$1$ and $x_i$. The important part to understand is, that we **replace** the original feature matrix:
$$
X_{ij} = 
\left[
\begin{array}{c}
x_1 \\ \dots \\ x_n
\end{array}
\right]
$$
into a **new feature matrix**, the one derived above.

Remember, we have only **one** feature, the $x$ coordinate, thus $j=1$.

2. ***Linear fit with polynomial***

When we want to fit a **polynomial function** through the data points, we are looking for a function:
$$
y_i = a_0 + a_1 x_i +a_2 x_i^2 + \dots + a_m x_i^m
$$
Our set of equations only becomes more tedious. For our $n$ observations, we can formulate the problem as:
$$
\left[
\begin{array}{c}
y_1 \\ y_2 \\ \vdots \\ y_n
\end{array}
\right]
=
\left[
\begin{array}{ccccc}
1 & x_1 & x_1^2 & \dots & x_1^m \\
1 & x_2 & x_2^2 & \dots & x_2^m\\
& \vdots \\
1 & x_n & x_n^2 & \dots & x_n^m
\end{array}
\right]
\left[
\begin{array}{c}
a_0 \\ a_1
\end{array}
\right]
$$
Again, we need to change our **original feature matrix** into a **new feature matrix** of coefficients of the
polynomial.

This is done by the method `PolynomialFeatures`. Let's play with it...

We have only **one** data point: $x_1=2$.
Our **original features matrix** $X_{old}$ is just the number $2$, wrapped into a matrix.
The **new feature matrix** $X_{new}$ is filled with the coefficients of the polynomial:
$(1,x,x^2,\dots,x^m)$. We have chosen $m=2$, a degree two polynomial.

Note that the last command, the `get_feature_names` method, return the order of the polynomial
coefficients.

In [ ]:
Xold = [[2]]
print(Xold)
poly = PolynomialFeatures(degree=3)
Xnew = poly.fit_transform(Xold)
print(Xnew)
poly.get_feature_names()

Take three observation points, the features $x_i=(1,2,10)$:

In [ ]:
Xold = [[1],[2],[10]]
print(Xold)
poly = PolynomialFeatures(degree=3)
Xnew = poly.fit_transform(Xold)
print(Xnew)
poly.get_feature_names()

Now we are ready to fit our data $(x_i,y_i)$ with a polynomial. We need **two steps**:
1. Convert the **original** feature matrix into the **polynomial coefficients** feature matrix (as above).
2. Run a **linear regession** method.

In [ ]:
X=x.reshape(n,1)
poly = PolynomialFeatures(degree=3)
X = poly.fit_transform(X)
linear_regression = LinearRegression()
linear_regression.fit(X,y)
#linear_regression.predict([[1,2,3,4]])

In [ ]:
x_test = np.linspace(-2,2,100).reshape(100,1)
fig=plt.figure(figsize=(12.0, 6.0))
plt.xlim([-2.1,2.1])
plt.ylim([-10,10])
plt.plot(x_test, f(x_test),linewidth=5,color='gray',label="true")
plt.plot(x_test, linear_regression.predict(poly.transform(x_test)),linewidth=2,label="Model")
plt.plot(x,y,marker='o',linestyle='none',color='red',label='data')
plt.legend()
plt.show()
print ('Score:     ',linear_regression.score(poly.transform(x_test),f(x_test)))

Instead our running two steps one after the other, we can stramline the process with a 
**pipeline**:

In [ ]:
X=x.reshape(n,1)
poly = PolynomialFeatures(degree=3)
linear_regression = LinearRegression()
pipeline = Pipeline([("polynomial_features", poly),
                     ("linear_regression", linear_regression)])
pipeline.fit(X, y)

In [ ]:
x_test = np.linspace(-2,2,100).reshape(100,1)
fig=plt.figure(figsize=(12.0, 6.0))
plt.xlim([-2.1,2.1])
plt.ylim([-10,10])
plt.plot(x_test, f(x_test),linewidth=5,color='gray',label="true")
plt.plot(x_test, pipeline.predict(x_test),linewidth=2,label="Model")
plt.plot(x,y,marker='o',linestyle='none',color='red',label='data')
plt.legend()
plt.show()
print ('Score:     ',pipeline.score(x_test,f(x_test)))

In [ ]:
pipeline.predict([[0.]])

[next>](04_regression3.ipynb)